# 第5章　数据处理（`Dataset` / `DataLoader`）

真实数据很大，无法整个塞进内存。PyTorch 用 **`Dataset`（如何取一条）** 和
**`DataLoader`（小批量・打乱・并行读取）** 高效供给数据。

本章目标：能写自定义 `Dataset`，用 `DataLoader` 跑批量，能读 `torchvision` 的图像数据。

> **本笔记使用方法**：从上到下 `Shift + Enter`。多数章节不需要 GPU；较重的章节会说明。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 5-1. 什么是小批量（mini-batch）？
不是一次用全部数据，而是分成**少量（例如32条）一份份**来训练，这就是小批量。
一次循环处理一个批量。把全部数据过一遍叫 **1 个 epoch**。

## 5-2. 自定义 `Dataset`
`Dataset` 只需实现两个方法：
- `__len__`：数据条数
- `__getitem__(i)`：返回第 i 条的 (输入, 标签)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, n=100):
        self.X = torch.randn(n, 3)               # 特征
        self.y = (self.X.sum(dim=1) > 0).long()  # 和为正则为1
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

ds = MyDataset(100)
print("条数:", len(ds))
print("第0条:", ds[0])

## 5-3. 用 `DataLoader` 分批
`batch_size` 是批大小，`shuffle=True` 每个 epoch 打乱顺序（训练时基本设 True）。

In [ ]:
loader = DataLoader(ds, batch_size=16, shuffle=True)

for xb, yb in loader:
    print("批量:", xb.shape, yb.shape)   # (16, 3), (16,)
    break   # 只看第一个批量

print("1个 epoch 的批数:", len(loader))   # 100/16 = 7 批

## 5-4. 图像数据：`torchvision` + `transforms`

`torchvision` 内置了 MNIST 等经典数据集和预处理 `transforms`。
- `transforms.ToTensor()`：把图像转成 `[0,1]` 的 Tensor，并整理成 `(C, H, W)`。
- `Normalize`：用均值・标准差归一化（训练更稳）。
- `Compose`：按顺序套用多个预处理。

> 首次会下载（几十 MB，需要联网）。Colab 上几秒。

In [ ]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),                 # (1,28,28), 值0..1
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST 的均值/标准差
])

train_ds = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

print("训练数据条数:", len(train_ds))
img, label = train_ds[0]
print("一张图的形状:", img.shape, " 标签:", label)   # (1,28,28), 5

In [ ]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=1000, shuffle=False)

xb, yb = next(iter(train_loader))
print("批量:", xb.shape, yb.shape)   # (64,1,28,28), (64,)

### 显示图像

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, i in zip(axes, range(6)):
    img, label = train_ds[i]
    ax.imshow(img.squeeze(), cmap="gray")   # (1,28,28)->(28,28)
    ax.set_title(str(label)); ax.axis("off")
plt.show()

## 5-5. 切分训练/验证集：`random_split`
把手头训练数据再切成 train / val，用来检查过拟合。

In [ ]:
from torch.utils.data import random_split
n_val = 10000
n_train = len(train_ds) - n_val
tr, va = random_split(train_ds, [n_train, n_val])
print("train:", len(tr), " val:", len(va))

## 练习 5
1. 把 `MyDataset` 的 `batch_size` 改成 8 / 64，确认 1 个 epoch 批数的变化。
2. 把 MNIST 图像显示更多张（例如16张）。
3. 给 `transforms` 加上 `transforms.RandomRotation(15)`，看图像怎么变？（数据增强入门）

In [ ]:
# 在这里写你自己的代码并运行
